In [1]:
# ============================================================
# 0) Install + imports
# ============================================================
!pip -q install gdown openpyxl

from openai import OpenAI
import pandas as pd
import gdown

# ============================================================
# 1) API client
# ============================================================
client = OpenAI(
    base_url="https://api.studio.nebius.ai/v1/",
    api_key="v1.CmMKHHN0YXRpY2tleS1lMDBwNGQwMGh0bnJqNzNoYmoSIXNlcnZpY2VhY2NvdW50LWUwMHk3OGVyOHduMzAwNGgxMTIMCJi8oskGEP3w3JoBOgsImL-6lAcQwNHhI0ACWgNlMDA.AAAAAAAAAAEJ7nTUTVdmJMmETh87kTvx2DfUm-5ajVh6WKbFQrgpvVUFGOKClqkg2r46AWvvQGQGi630cd9BR1olXWpt7g4E"
)

# ============================================================
# 2) Download dataset
# ============================================================
file_id = "1TF2LJiUB_Db2Srb-PF-oM1tWwLzN0T6P"
out_path = "/content/dataset.xlsx"
gdown.download(id=file_id, output=out_path, quiet=True)

# ============================================================
# 3) Load data
# ============================================================
df = pd.read_excel(out_path, dtype=str)
df.columns = [c.strip() for c in df.columns]

# Auto-detect columns
col_mapping = {}
for col in df.columns:
    col_lower = col.lower()
    if 'id' in col_lower and 'sentence' in col_lower:
        col_mapping[col] = 'sentence_id'
    elif 'sentence' in col_lower and 'id' not in col_lower:
        col_mapping[col] = 'sentence'
    elif 'annotation' in col_lower or 'term' in col_lower:
        col_mapping[col] = 'annotations'

if col_mapping:
    df = df.rename(columns=col_mapping)

df = df[df["sentence"].notna() & (df["sentence"] != "")]
df = df.reset_index(drop=True)

# ============================================================
# 4) USER INPUT - Select rows to process
# ============================================================
print(f"Total rows available: {len(df)}\n")

from_row = int(input("From row (starting from 0): "))
to_row = int(input("To row (inclusive): "))

# Validate input
if from_row < 0 or to_row >= len(df) or from_row > to_row:
    raise ValueError(f"Invalid range. Must be between 0 and {len(df)-1}")

# Select the range
df = df.iloc[from_row:to_row+1].copy()
df = df.reset_index(drop=True)

# ============================================================
# 5) Add output columns
# ============================================================
df["Prompt_A"] = ""
df["Prompt_B"] = ""
df["Prompt_C"] = ""

# ============================================================
# 6) Prompts
# ============================================================
prompt_A = (
    "You are an expert in medical terminology extraction. "
    "Extract all medical terms from the given sentence. "
    "Return ONLY the result in the format [term1, term2, ...]."
)

prompt_B = (
    "You are an expert in medical terminology extraction.\n\n"
    "Here are examples:\n"
    'Sentence: "The patient presented with acute myocardial infarction."\n'
    "Terms: [acute myocardial infarction]\n\n"
    'Sentence: "Treatment includes beta blockers and ACE inhibitors."\n'
    "Terms: [beta blockers, ACE inhibitors]\n\n"
    "Now extract all medical terms from the following sentence. "
    "Make no explanations and only output [term1, term2, ...]."
)

prompt_C = (
    "Extract medical and clinical terms from the sentence. "
    "Respond strictly as a Python-style list: [term1, term2, ...]."
)

MODEL_NAME = "GPT-3.5"
TEMPERATURE = 0.0

# ============================================================
# 7) Run inference (silent)
# ============================================================
for idx, row in df.iterrows():
    sentence = str(row["sentence"]).strip()

    for prompt, col in [(prompt_A, "Prompt_A"), (prompt_B, "Prompt_B"), (prompt_C, "Prompt_C")]:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            temperature=TEMPERATURE,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": sentence},
            ],
        )
        df.at[idx, col] = response.choices[0].message.content.strip()

# ============================================================
# 8) Final DataFrame (no prints, just show df)
# ============================================================
final_df = df[["sentence_id", "sentence", "annotations", "Prompt_A", "Prompt_B", "Prompt_C"]]

# Save to Excel
final_df.to_excel("results.xlsx", index=False)

# Display
final_df


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


ModuleNotFoundError: No module named 'openai'

In [ ]:
# ============================================================
# Save as CSV with row range in filename
# ============================================================
csv_filename = f"results_row_{from_row}_to_{to_row}.csv"
final_df.to_csv(csv_filename, index=False)
print(f"✓ Saved to: {csv_filename}")

In [ ]:
# ============================================================
# EVALUATION METRICS FOR TERM EXTRACTION
# ============================================================
import pandas as pd
import re
from sklearn.metrics import precision_score, recall_score, f1_score

def parse_terms(term_string):
    """
    Parse terms from different formats:
    - "term1, term2, term3"
    - "[term1, term2, term3]"
    - "['term1', 'term2', 'term3']"
    """
    if pd.isna(term_string) or term_string == "" or term_string == "[]":
        return set()

    # Remove brackets and quotes
    term_string = str(term_string).strip()
    term_string = re.sub(r'[\[\]\'"]+', '', term_string)

    # Split by comma and clean
    terms = [t.strip().lower() for t in term_string.split(',')]
    terms = [t for t in terms if t and t != 'nan']

    return set(terms)

def calculate_metrics(predicted_terms, ground_truth_terms):
    """
    Calculate Precision, Recall, F1-Score
    """
    if len(ground_truth_terms) == 0 and len(predicted_terms) == 0:
        return 1.0, 1.0, 1.0  # Perfect if both empty

    if len(ground_truth_terms) == 0:
        return 0.0, 0.0, 0.0  # No ground truth

    if len(predicted_terms) == 0:
        return 0.0, 0.0, 0.0  # No predictions

    # True Positives: terms in both predicted and ground truth
    true_positives = len(predicted_terms & ground_truth_terms)

    # Precision: TP / (TP + FP)
    precision = true_positives / len(predicted_terms) if len(predicted_terms) > 0 else 0.0

    # Recall: TP / (TP + FN)
    recall = true_positives / len(ground_truth_terms) if len(ground_truth_terms) > 0 else 0.0

    # F1-Score: Harmonic mean
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    return precision, recall, f1

# ============================================================
# Load your results (adjust filename if needed)
# ============================================================
# If you just ran the previous cell, final_df is already in memory
# Otherwise, load from CSV:
# final_df = pd.read_csv("results_row_0_to_9.csv")

print("="*70)
print("EVALUATION METRICS - Term Extraction")
print("="*70)

# ============================================================
# Calculate metrics for each prompt
# ============================================================
results = []

for idx, row in final_df.iterrows():
    # Parse ground truth
    ground_truth = parse_terms(row['annotations'])

    # Parse predictions
    prompt_a_terms = parse_terms(row['Prompt_A'])
    prompt_b_terms = parse_terms(row['Prompt_B'])
    prompt_c_terms = parse_terms(row['Prompt_C'])

    # Calculate metrics
    p_a, r_a, f1_a = calculate_metrics(prompt_a_terms, ground_truth)
    p_b, r_b, f1_b = calculate_metrics(prompt_b_terms, ground_truth)
    p_c, r_c, f1_c = calculate_metrics(prompt_c_terms, ground_truth)

    results.append({
        'sentence_id': row['sentence_id'],
        'num_ground_truth': len(ground_truth),

        'Prompt_A_Precision': p_a,
        'Prompt_A_Recall': r_a,
        'Prompt_A_F1': f1_a,
        'Prompt_A_Count': len(prompt_a_terms),

        'Prompt_B_Precision': p_b,
        'Prompt_B_Recall': r_b,
        'Prompt_B_F1': f1_b,
        'Prompt_B_Count': len(prompt_b_terms),

        'Prompt_C_Precision': p_c,
        'Prompt_C_Recall': r_c,
        'Prompt_C_F1': f1_c,
        'Prompt_C_Count': len(prompt_c_terms),
    })

metrics_df = pd.DataFrame(results)

# ============================================================
# Overall Average Metrics
# ============================================================
print("\n📊 OVERALL AVERAGE METRICS:")
print("-" * 70)

avg_metrics = {
    'Prompt_A': {
        'Precision': metrics_df['Prompt_A_Precision'].mean(),
        'Recall': metrics_df['Prompt_A_Recall'].mean(),
        'F1-Score': metrics_df['Prompt_A_F1'].mean()
    },
    'Prompt_B': {
        'Precision': metrics_df['Prompt_B_Precision'].mean(),
        'Recall': metrics_df['Prompt_B_Recall'].mean(),
        'F1-Score': metrics_df['Prompt_B_F1'].mean()
    },
    'Prompt_C': {
        'Precision': metrics_df['Prompt_C_Precision'].mean(),
        'Recall': metrics_df['Prompt_C_Recall'].mean(),
        'F1-Score': metrics_df['Prompt_C_F1'].mean()
    }
}

avg_df = pd.DataFrame(avg_metrics).T
print(avg_df.to_string())

# ============================================================
# Best Performing Prompt
# ============================================================
print("\n🏆 BEST PERFORMING PROMPT (by F1-Score):")
print("-" * 70)
best_prompt = avg_df['F1-Score'].idxmax()
best_f1 = avg_df.loc[best_prompt, 'F1-Score']
print(f"{best_prompt}: F1-Score = {best_f1:.4f}")

# ============================================================
# Per-Sentence Metrics
# ============================================================
print("\n📋 PER-SENTENCE METRICS:")
print("-" * 70)
print(metrics_df.to_string())

# ============================================================
# Save metrics to CSV
# ============================================================
metrics_filename = f"evaluation_metrics_row_{from_row}_to_{to_row}.csv"
metrics_df.to_csv(metrics_filename, index=False)
print(f"\n✓ Saved detailed metrics to: {metrics_filename}")

# ============================================================
# Summary Statistics
# ============================================================
print("\n📈 SUMMARY STATISTICS:")
print("-" * 70)
print(f"Total sentences evaluated: {len(metrics_df)}")
print(f"Average ground truth terms per sentence: {metrics_df['num_ground_truth'].mean():.2f}")
print(f"\nAverage extracted terms per sentence:")
print(f"  Prompt_A: {metrics_df['Prompt_A_Count'].mean():.2f}")
print(f"  Prompt_B: {metrics_df['Prompt_B_Count'].mean():.2f}")
print(f"  Prompt_C: {metrics_df['Prompt_C_Count'].mean():.2f}")

# Display the metrics dataframe
metrics_df

In [ ]:
# ============================================================
# OPTIONAL: VISUALIZE METRICS
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# ============================================================
# 1) Bar Chart: Average Metrics Comparison
# ============================================================
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

x = ['Prompt_A', 'Prompt_B', 'Prompt_C']
precision = [avg_metrics[p]['Precision'] for p in x]
recall = [avg_metrics[p]['Recall'] for p in x]
f1 = [avg_metrics[p]['F1-Score'] for p in x]

x_pos = range(len(x))
width = 0.25

ax.bar([p - width for p in x_pos], precision, width, label='Precision', color='#3498db')
ax.bar(x_pos, recall, width, label='Recall', color='#2ecc71')
ax.bar([p + width for p in x_pos], f1, width, label='F1-Score', color='#e74c3c')

ax.set_xlabel('Prompt Type', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Term Extraction Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(x)
ax.legend()
ax.set_ylim([0, 1.0])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'metrics_comparison_row_{from_row}_to_{to_row}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved chart to: metrics_comparison_row_{from_row}_to_{to_row}.png")

# ============================================================
# 2) Heatmap: Per-Sentence F1 Scores
# ============================================================
fig, ax = plt.subplots(1, 1, figsize=(10, max(6, len(metrics_df) * 0.4)))

heatmap_data = metrics_df[['sentence_id', 'Prompt_A_F1', 'Prompt_B_F1', 'Prompt_C_F1']].set_index('sentence_id')
heatmap_data.columns = ['Prompt A', 'Prompt B', 'Prompt C']

sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1,
            cbar_kws={'label': 'F1-Score'}, linewidths=0.5)

ax.set_title('F1-Score Heatmap per Sentence', fontsize=14, fontweight='bold')
ax.set_xlabel('Prompt Type', fontsize=12, fontweight='bold')
ax.set_ylabel('Sentence ID', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(f'heatmap_row_{from_row}_to_{to_row}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved heatmap to: heatmap_row_{from_row}_to_{to_row}.png")

# ============================================================
# 3) Line Plot: Metrics Across Sentences
# ============================================================
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

ax.plot(metrics_df.index, metrics_df['Prompt_A_F1'], marker='o', label='Prompt A', linewidth=2)
ax.plot(metrics_df.index, metrics_df['Prompt_B_F1'], marker='s', label='Prompt B', linewidth=2)
ax.plot(metrics_df.index, metrics_df['Prompt_C_F1'], marker='^', label='Prompt C', linewidth=2)

ax.set_xlabel('Sentence Index', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax.set_title('F1-Score Across Sentences', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1.0])

plt.tight_layout()
plt.savefig(f'f1_trend_row_{from_row}_to_{to_row}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved trend chart to: f1_trend_row_{from_row}_to_{to_row}.png")

In [ ]:
# ============================================================
# UNDERSTANDING THE EVALUATION METRICS
# ============================================================

"""
WHAT DO THE METRICS MEAN?
--------------------------

1. PRECISION
   - "Of all the terms the model extracted, how many were actually correct?"
   - Formula: True Positives / (True Positives + False Positives)
   - High precision = Model doesn't extract many wrong terms
   - Range: 0.0 to 1.0 (higher is better)

   Example:
   Ground truth: [heart failure, hypertension]
   Model output: [heart failure, diabetes]
   Precision = 1/2 = 0.5 (1 correct out of 2 extracted)

2. RECALL
   - "Of all the correct terms, how many did the model find?"
   - Formula: True Positives / (True Positives + False Negatives)
   - High recall = Model doesn't miss many correct terms
   - Range: 0.0 to 1.0 (higher is better)

   Example:
   Ground truth: [heart failure, hypertension]
   Model output: [heart failure, diabetes]
   Recall = 1/2 = 0.5 (found 1 out of 2 ground truth terms)

3. F1-SCORE
   - Harmonic mean of Precision and Recall
   - Formula: 2 * (Precision * Recall) / (Precision + Recall)
   - Balances precision and recall
   - Range: 0.0 to 1.0 (higher is better)
   - This is typically the MAIN metric to compare prompts

   Example from above:
   F1 = 2 * (0.5 * 0.5) / (0.5 + 0.5) = 0.5


REAL EXAMPLE FROM YOUR DATA:
----------------------------

Sentence: "The patient presented with acute myocardial infarction."

Ground Truth (Human): [acute myocardial infarction]
Prompt A Output: [acute myocardial infarction, patient]
Prompt B Output: [acute myocardial infarction]
Prompt C Output: []

Metrics:
--------
Prompt A:
  - True Positives: 1 (acute myocardial infarction)
  - False Positives: 1 (patient - not in ground truth)
  - False Negatives: 0 (no terms missed)
  - Precision = 1/2 = 0.50
  - Recall = 1/1 = 1.00
  - F1 = 2 * (0.50 * 1.00) / (0.50 + 1.00) = 0.67

Prompt B:
  - True Positives: 1
  - False Positives: 0
  - False Negatives: 0
  - Precision = 1/1 = 1.00
  - Recall = 1/1 = 1.00
  - F1 = 1.00 (PERFECT!)

Prompt C:
  - True Positives: 0
  - False Positives: 0
  - False Negatives: 1 (missed the term)
  - Precision = 0.00
  - Recall = 0.00
  - F1 = 0.00


WHICH PROMPT IS BEST?
----------------------

Look at the AVERAGE F1-SCORE across all sentences:
  - Highest F1-Score = Best overall performance
  - Consider the trade-off:
    * High Precision, Low Recall = Conservative (misses terms)
    * Low Precision, High Recall = Liberal (extracts too many)
    * Balanced F1 = Good middle ground


WHAT TO DO WITH RESULTS?
-------------------------

1. If one prompt clearly outperforms:
   ✓ Use that prompt for future extractions

2. If prompts are similar:
   ✓ Look at per-sentence metrics
   ✓ See which prompt is more consistent
   ✓ Consider which errors are more acceptable (false positives vs false negatives)

3. If all prompts perform poorly:
   ✓ Refine your prompts
   ✓ Add more examples
   ✓ Try a different model
   ✓ Check if ground truth annotations are consistent


TYPICAL GOOD SCORES:
--------------------
F1 > 0.80  →  Excellent
F1 > 0.70  →  Good
F1 > 0.60  →  Acceptable
F1 < 0.50  →  Needs improvement
"""

print(__doc__)

In [ ]:
# ============================================================
# CREATE EVALUATION METRICS SUMMARY TABLE (IMAGE)
# ============================================================
import pandas as pd
import re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def parse_terms(term_string):
    """Parse terms from different formats"""
    if pd.isna(term_string) or term_string == "" or term_string == "[]":
        return set()

    term_string = str(term_string).strip()
    term_string = re.sub(r'[\[\]\'"]+', '', term_string)
    terms = [t.strip().lower() for t in term_string.split(',')]
    terms = [t for t in terms if t and t != 'nan']

    return set(terms)

def calculate_metrics(predicted_terms, ground_truth_terms):
    """Calculate Precision, Recall, F1"""
    if len(ground_truth_terms) == 0 and len(predicted_terms) == 0:
        return 1.0, 1.0, 1.0
    if len(ground_truth_terms) == 0 or len(predicted_terms) == 0:
        return 0.0, 0.0, 0.0

    true_positives = len(predicted_terms & ground_truth_terms)
    precision = true_positives / len(predicted_terms) if len(predicted_terms) > 0 else 0.0
    recall = true_positives / len(ground_truth_terms) if len(ground_truth_terms) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    return precision, recall, f1

# ============================================================
# Calculate metrics for each prompt
# ============================================================
results = []

for idx, row in final_df.iterrows():
    ground_truth = parse_terms(row['annotations'])

    prompt_a_terms = parse_terms(row['Prompt_A'])
    prompt_b_terms = parse_terms(row['Prompt_B'])
    prompt_c_terms = parse_terms(row['Prompt_C'])

    p_a, r_a, f1_a = calculate_metrics(prompt_a_terms, ground_truth)
    p_b, r_b, f1_b = calculate_metrics(prompt_b_terms, ground_truth)
    p_c, r_c, f1_c = calculate_metrics(prompt_c_terms, ground_truth)

    results.append({
        'Prompt_A_Precision': p_a,
        'Prompt_A_Recall': r_a,
        'Prompt_A_F1': f1_a,
        'Prompt_A_Count': len(prompt_a_terms),

        'Prompt_B_Precision': p_b,
        'Prompt_B_Recall': r_b,
        'Prompt_B_F1': f1_b,
        'Prompt_B_Count': len(prompt_b_terms),

        'Prompt_C_Precision': p_c,
        'Prompt_C_Recall': r_c,
        'Prompt_C_F1': f1_c,
        'Prompt_C_Count': len(prompt_c_terms),

        'Ground_Truth_Count': len(ground_truth)
    })

metrics_df = pd.DataFrame(results)

# ============================================================
# Calculate average metrics
# ============================================================
avg_metrics = {
    'Prompt A': [
        metrics_df['Prompt_A_Precision'].mean(),
        metrics_df['Prompt_A_Recall'].mean(),
        metrics_df['Prompt_A_F1'].mean(),
        metrics_df['Prompt_A_Count'].mean()
    ],
    'Prompt B': [
        metrics_df['Prompt_B_Precision'].mean(),
        metrics_df['Prompt_B_Recall'].mean(),
        metrics_df['Prompt_B_F1'].mean(),
        metrics_df['Prompt_B_Count'].mean()
    ],
    'Prompt C': [
        metrics_df['Prompt_C_Precision'].mean(),
        metrics_df['Prompt_C_Recall'].mean(),
        metrics_df['Prompt_C_F1'].mean(),
        metrics_df['Prompt_C_Count'].mean()
    ]
}

# ============================================================
# Create the summary table as image
# ============================================================
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('tight')
ax.axis('off')

# Prepare data for table
table_data = []
table_data.append(['Metric', 'Prompt A', 'Prompt B', 'Prompt C'])
table_data.append(['Precision',
                   f"{avg_metrics['Prompt A'][0]:.4f}",
                   f"{avg_metrics['Prompt B'][0]:.4f}",
                   f"{avg_metrics['Prompt C'][0]:.4f}"])
table_data.append(['Recall',
                   f"{avg_metrics['Prompt A'][1]:.4f}",
                   f"{avg_metrics['Prompt B'][1]:.4f}",
                   f"{avg_metrics['Prompt C'][1]:.4f}"])
table_data.append(['F1-Score',
                   f"{avg_metrics['Prompt A'][2]:.4f}",
                   f"{avg_metrics['Prompt B'][2]:.4f}",
                   f"{avg_metrics['Prompt C'][2]:.4f}"])
table_data.append(['Avg Terms Extracted',
                   f"{avg_metrics['Prompt A'][3]:.2f}",
                   f"{avg_metrics['Prompt B'][3]:.2f}",
                   f"{avg_metrics['Prompt C'][3]:.2f}"])

# Find best F1 score
best_f1_idx = max(range(3), key=lambda i: avg_metrics[['Prompt A', 'Prompt B', 'Prompt C'][i]][2])

# Color cells based on best performance
cell_colors = []
for i in range(len(table_data)):
    if i == 0:  # Header row
        cell_colors.append(['#3498db', '#3498db', '#3498db', '#3498db'])
    elif i == 3:  # F1-Score row - highlight best
        row_colors = ['#f8f9fa', '#f8f9fa', '#f8f9fa', '#f8f9fa']
        row_colors[best_f1_idx + 1] = '#2ecc71'  # Green for best
        cell_colors.append(row_colors)
    else:
        cell_colors.append(['#f8f9fa', 'white', 'white', 'white'])

# Create table
table = ax.table(cellText=table_data, cellLoc='center', loc='center',
                cellColours=cell_colors, bbox=[0, 0, 1, 1])

table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 2.5)

# Style header row
for i in range(4):
    table[(0, i)].set_text_props(weight='bold', color='white')

# Style metric names column
for i in range(1, len(table_data)):
    table[(i, 0)].set_text_props(weight='bold')

# Add title
title_text = f"Evaluation Metrics Summary\nRows {from_row} to {to_row} ({to_row - from_row + 1} sentences)"
plt.figtext(0.5, 0.95, title_text, ha='center', fontsize=16, weight='bold')

# Add footer with additional info
footer_text = (f"Ground Truth Avg: {metrics_df['Ground_Truth_Count'].mean():.2f} terms/sentence | "
               f"Best Performing: Prompt {chr(65 + best_f1_idx)} (F1={avg_metrics[['Prompt A', 'Prompt B', 'Prompt C'][best_f1_idx]][2]:.4f})")
plt.figtext(0.5, 0.05, footer_text, ha='center', fontsize=10, style='italic', color='gray')

# Save the image
img_filename = f"evaluation_summary_row_{from_row}_to_{to_row}.png"
plt.savefig(img_filename, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("="*70)
print("📊 EVALUATION SUMMARY")
print("="*70)
print(f"✓ Saved summary table to: {img_filename}")
print(f"\n🏆 Best Performing Prompt: Prompt {chr(65 + best_f1_idx)}")
print(f"   F1-Score: {avg_metrics[['Prompt A', 'Prompt B', 'Prompt C'][best_f1_idx]][2]:.4f}")
print(f"\n📈 Detailed metrics:")
for prompt_name, metrics in avg_metrics.items():
    print(f"   {prompt_name}: P={metrics[0]:.4f}, R={metrics[1]:.4f}, F1={metrics[2]:.4f}")

In [ ]:
# ============================================================
# CREATE COMPREHENSIVE EVALUATION SUMMARY TABLE (IMAGE)
# ============================================================
import pandas as pd
import re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from textwrap import fill

def parse_terms(term_string):
    """Parse terms from different formats"""
    if pd.isna(term_string) or term_string == "" or term_string == "[]":
        return set()

    term_string = str(term_string).strip()
    term_string = re.sub(r'[\[\]\'"]+', '', term_string)
    terms = [t.strip().lower() for t in term_string.split(',')]
    terms = [t for t in terms if t and t != 'nan']

    return set(terms)

def calculate_metrics(predicted_terms, ground_truth_terms):
    """Calculate Precision, Recall, F1"""
    if len(ground_truth_terms) == 0 and len(predicted_terms) == 0:
        return 1.0, 1.0, 1.0
    if len(ground_truth_terms) == 0 or len(predicted_terms) == 0:
        return 0.0, 0.0, 0.0

    true_positives = len(predicted_terms & ground_truth_terms)
    precision = true_positives / len(predicted_terms) if len(predicted_terms) > 0 else 0.0
    recall = true_positives / len(ground_truth_terms) if len(ground_truth_terms) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    return precision, recall, f1

# ============================================================
# Calculate metrics
# ============================================================
results = []

for idx, row in final_df.iterrows():
    ground_truth = parse_terms(row['annotations'])

    prompt_a_terms = parse_terms(row['Prompt_A'])
    prompt_b_terms = parse_terms(row['Prompt_B'])
    prompt_c_terms = parse_terms(row['Prompt_C'])

    p_a, r_a, f1_a = calculate_metrics(prompt_a_terms, ground_truth)
    p_b, r_b, f1_b = calculate_metrics(prompt_b_terms, ground_truth)
    p_c, r_c, f1_c = calculate_metrics(prompt_c_terms, ground_truth)

    results.append({
        'Prompt_A_Precision': p_a, 'Prompt_A_Recall': r_a, 'Prompt_A_F1': f1_a, 'Prompt_A_Count': len(prompt_a_terms),
        'Prompt_B_Precision': p_b, 'Prompt_B_Recall': r_b, 'Prompt_B_F1': f1_b, 'Prompt_B_Count': len(prompt_b_terms),
        'Prompt_C_Precision': p_c, 'Prompt_C_Recall': r_c, 'Prompt_C_F1': f1_c, 'Prompt_C_Count': len(prompt_c_terms),
        'Ground_Truth_Count': len(ground_truth)
    })

metrics_df = pd.DataFrame(results)

# Calculate averages
avg_metrics = {
    'Prompt A': [
        metrics_df['Prompt_A_Precision'].mean(),
        metrics_df['Prompt_A_Recall'].mean(),
        metrics_df['Prompt_A_F1'].mean(),
        metrics_df['Prompt_A_Count'].mean()
    ],
    'Prompt B': [
        metrics_df['Prompt_B_Precision'].mean(),
        metrics_df['Prompt_B_Recall'].mean(),
        metrics_df['Prompt_B_F1'].mean(),
        metrics_df['Prompt_B_Count'].mean()
    ],
    'Prompt C': [
        metrics_df['Prompt_C_Precision'].mean(),
        metrics_df['Prompt_C_Recall'].mean(),
        metrics_df['Prompt_C_F1'].mean(),
        metrics_df['Prompt_C_Count'].mean()
    ]
}

# ============================================================
# Create comprehensive figure with multiple sections
# ============================================================
fig = plt.figure(figsize=(16, 14))

# Create grid
gs = fig.add_gridspec(3, 1, height_ratios=[1.5, 3, 2], hspace=0.4)

# ============================================================
# SECTION 1: Configuration Info
# ============================================================
ax1 = fig.add_subplot(gs[0])
ax1.axis('off')

config_text = f"""
MODEL: {MODEL_NAME}
TEMPERATURE: {TEMPERATURE}
ROWS PROCESSED: {from_row} to {to_row} ({to_row - from_row + 1} sentences)
GROUND TRUTH AVG: {metrics_df['Ground_Truth_Count'].mean():.2f} terms/sentence
"""

ax1.text(0.5, 0.5, config_text.strip(), ha='center', va='center',
         fontsize=12, family='monospace',
         bbox=dict(boxstyle='round', facecolor='#ecf0f1', edgecolor='#34495e', linewidth=2))

# ============================================================
# SECTION 2: Metrics Table
# ============================================================
ax2 = fig.add_subplot(gs[1])
ax2.axis('tight')
ax2.axis('off')

# Prepare data
table_data = []
table_data.append(['Metric', 'Prompt A', 'Prompt B', 'Prompt C'])
table_data.append(['Precision',
                   f"{avg_metrics['Prompt A'][0]:.4f}",
                   f"{avg_metrics['Prompt B'][0]:.4f}",
                   f"{avg_metrics['Prompt C'][0]:.4f}"])
table_data.append(['Recall',
                   f"{avg_metrics['Prompt A'][1]:.4f}",
                   f"{avg_metrics['Prompt B'][1]:.4f}",
                   f"{avg_metrics['Prompt C'][1]:.4f}"])
table_data.append(['F1-Score',
                   f"{avg_metrics['Prompt A'][2]:.4f}",
                   f"{avg_metrics['Prompt B'][2]:.4f}",
                   f"{avg_metrics['Prompt C'][2]:.4f}"])
table_data.append(['Avg Terms',
                   f"{avg_metrics['Prompt A'][3]:.2f}",
                   f"{avg_metrics['Prompt B'][3]:.2f}",
                   f"{avg_metrics['Prompt C'][3]:.2f}"])

# Find best F1
best_f1_idx = max(range(3), key=lambda i: avg_metrics[['Prompt A', 'Prompt B', 'Prompt C'][i]][2])

# Color cells
cell_colors = []
for i in range(len(table_data)):
    if i == 0:
        cell_colors.append(['#3498db', '#3498db', '#3498db', '#3498db'])
    elif i == 3:  # F1 row
        row_colors = ['#f8f9fa', '#f8f9fa', '#f8f9fa', '#f8f9fa']
        row_colors[best_f1_idx + 1] = '#2ecc71'
        cell_colors.append(row_colors)
    else:
        cell_colors.append(['#f8f9fa', 'white', 'white', 'white'])

table = ax2.table(cellText=table_data, cellLoc='center', loc='center',
                 cellColours=cell_colors, bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(14)
table.scale(1, 3)

for i in range(4):
    table[(0, i)].set_text_props(weight='bold', color='white', fontsize=14)
for i in range(1, len(table_data)):
    table[(i, 0)].set_text_props(weight='bold', fontsize=13)

# ============================================================
# SECTION 3: Prompts Text
# ============================================================
ax3 = fig.add_subplot(gs[2])
ax3.axis('off')

# Get the actual prompts used
prompt_a_text = prompt_A
prompt_b_text = prompt_B
prompt_c_text = prompt_C

# Truncate if too long
max_len = 200
if len(prompt_a_text) > max_len:
    prompt_a_text = prompt_a_text[:max_len] + "..."
if len(prompt_b_text) > max_len:
    prompt_b_text = prompt_b_text[:max_len] + "..."
if len(prompt_c_text) > max_len:
    prompt_c_text = prompt_c_text[:max_len] + "..."

prompts_text = f"""PROMPT A:
{fill(prompt_a_text, width=120)}

PROMPT B:
{fill(prompt_b_text, width=120)}

PROMPT C:
{fill(prompt_c_text, width=120)}
"""

ax3.text(0.05, 0.95, prompts_text.strip(), ha='left', va='top',
         fontsize=9, family='monospace',
         bbox=dict(boxstyle='round', facecolor='#fef9e7', edgecolor='#f39c12', linewidth=1.5))

# ============================================================
# Add title and footer
# ============================================================
fig.suptitle('Evaluation Metrics Summary', fontsize=20, weight='bold', y=0.98)

footer_text = f"🏆 Best Performing: Prompt {chr(65 + best_f1_idx)} (F1={avg_metrics[['Prompt A', 'Prompt B', 'Prompt C'][best_f1_idx]][2]:.4f})"
fig.text(0.5, 0.02, footer_text, ha='center', fontsize=12, weight='bold', color='#27ae60')

# Save
img_filename = f"evaluation_summary_row_{from_row}_to_{to_row}.png"
plt.savefig(img_filename, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("="*70)
print("📊 COMPREHENSIVE EVALUATION SUMMARY")
print("="*70)
print(f"✓ Saved summary to: {img_filename}")
print(f"\n🏆 Best Performing: Prompt {chr(65 + best_f1_idx)}")
print(f"   F1-Score: {avg_metrics[['Prompt A', 'Prompt B', 'Prompt C'][best_f1_idx]][2]:.4f}")
print(f"\n📈 All metrics:")
for prompt_name, metrics in avg_metrics.items():
    print(f"   {prompt_name}: P={metrics[0]:.4f}, R={metrics[1]:.4f}, F1={metrics[2]:.4f}")

In [ ]:
# ============================================================
# IMAGE 1: DATA TABLE WITH RESULTS
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.table import Table

# Prepare data for display (without sentence_id)
display_df = final_df[['sentence', 'annotations', 'Prompt_A', 'Prompt_B', 'Prompt_C']].copy()
display_df.columns = ['Sentence', 'Human_Annotations', 'Prompt_A', 'Prompt_B', 'Prompt_C']

# Function to wrap text
def wrap_text(text, width=50):
    """Wrap text to fit in cells"""
    if pd.isna(text):
        return ""
    text = str(text)
    words = text.split()
    lines = []
    current_line = []
    current_length = 0

    for word in words:
        if current_length + len(word) + 1 <= width:
            current_line.append(word)
            current_length += len(word) + 1
        else:
            if current_line:
                lines.append(' '.join(current_line))
            current_line = [word]
            current_length = len(word)

    if current_line:
        lines.append(' '.join(current_line))

    return '\n'.join(lines)

# Apply text wrapping
for col in display_df.columns:
    display_df[col] = display_df[col].apply(lambda x: wrap_text(x, width=40))

# Calculate figure height based on number of rows
num_rows = len(display_df)
row_height = 1.2
fig_height = max(10, num_rows * row_height + 2)

# Create figure
fig, ax = plt.subplots(figsize=(20, fig_height))
ax.axis('tight')
ax.axis('off')

# Prepare table data
table_data = [display_df.columns.tolist()] + display_df.values.tolist()

# Create color scheme
header_color = '#3498db'
row_colors = ['#f8f9fa' if i % 2 == 0 else 'white' for i in range(len(display_df))]
cell_colors = [[header_color] * 5] + [[color] * 5 for color in row_colors]

# Create table
table = ax.table(cellText=table_data, cellLoc='left', loc='center',
                cellColours=cell_colors, bbox=[0, 0, 1, 1])

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2)

# Style header
for i in range(5):
    cell = table[(0, i)]
    cell.set_text_props(weight='bold', color='white', fontsize=10)
    cell.set_height(0.05)

# Set column widths
widths = [0.30, 0.15, 0.15, 0.15, 0.15]  # Sentence wider, others equal
for i, width in enumerate(widths):
    for j in range(len(table_data)):
        table[(j, i)].set_width(width)

# Add title
title = f"Term Extraction Results - Rows {from_row} to {to_row}"
plt.figtext(0.5, 0.98, title, ha='center', fontsize=16, weight='bold')

# Save
img1_filename = f"data_table_row_{from_row}_to_{to_row}.png"
plt.savefig(img1_filename, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(f"✓ Image 1 saved: {img1_filename}")

# ============================================================
# IMAGE 2: EVALUATION METRICS WITH FULL PROMPTS
# ============================================================
import re

def parse_terms(term_string):
    """Parse terms from different formats"""
    if pd.isna(term_string) or term_string == "" or term_string == "[]":
        return set()

    term_string = str(term_string).strip()
    term_string = re.sub(r'[\[\]\'"]+', '', term_string)
    terms = [t.strip().lower() for t in term_string.split(',')]
    terms = [t for t in terms if t and t != 'nan']

    return set(terms)

def calculate_metrics(predicted_terms, ground_truth_terms):
    """Calculate Precision, Recall, F1"""
    if len(ground_truth_terms) == 0 and len(predicted_terms) == 0:
        return 1.0, 1.0, 1.0
    if len(ground_truth_terms) == 0 or len(predicted_terms) == 0:
        return 0.0, 0.0, 0.0

    true_positives = len(predicted_terms & ground_truth_terms)
    precision = true_positives / len(predicted_terms) if len(predicted_terms) > 0 else 0.0
    recall = true_positives / len(ground_truth_terms) if len(ground_truth_terms) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    return precision, recall, f1

# Calculate metrics
results = []
for idx, row in final_df.iterrows():
    ground_truth = parse_terms(row['annotations'])
    prompt_a_terms = parse_terms(row['Prompt_A'])
    prompt_b_terms = parse_terms(row['Prompt_B'])
    prompt_c_terms = parse_terms(row['Prompt_C'])

    p_a, r_a, f1_a = calculate_metrics(prompt_a_terms, ground_truth)
    p_b, r_b, f1_b = calculate_metrics(prompt_b_terms, ground_truth)
    p_c, r_c, f1_c = calculate_metrics(prompt_c_terms, ground_truth)

    results.append({
        'Prompt_A_Precision': p_a, 'Prompt_A_Recall': r_a, 'Prompt_A_F1': f1_a,
        'Prompt_B_Precision': p_b, 'Prompt_B_Recall': r_b, 'Prompt_B_F1': f1_b,
        'Prompt_C_Precision': p_c, 'Prompt_C_Recall': r_c, 'Prompt_C_F1': f1_c,
        'Ground_Truth_Count': len(ground_truth)
    })

metrics_df = pd.DataFrame(results)

# Calculate averages
avg_metrics = {
    'Prompt A': [
        metrics_df['Prompt_A_Precision'].mean(),
        metrics_df['Prompt_A_Recall'].mean(),
        metrics_df['Prompt_A_F1'].mean()
    ],
    'Prompt B': [
        metrics_df['Prompt_B_Precision'].mean(),
        metrics_df['Prompt_B_Recall'].mean(),
        metrics_df['Prompt_B_F1'].mean()
    ],
    'Prompt C': [
        metrics_df['Prompt_C_Precision'].mean(),
        metrics_df['Prompt_C_Recall'].mean(),
        metrics_df['Prompt_C_F1'].mean()
    ]
}

best_f1_idx = max(range(3), key=lambda i: avg_metrics[['Prompt A', 'Prompt B', 'Prompt C'][i]][2])

# Create figure with sections
fig = plt.figure(figsize=(18, 16))
gs = fig.add_gridspec(4, 1, height_ratios=[1, 2, 4, 1], hspace=0.3)

# ============================================================
# SECTION 1: Model Configuration
# ============================================================
ax1 = fig.add_subplot(gs[0])
ax1.axis('off')

config_text = f"""MODEL: {MODEL_NAME}  |  TEMPERATURE: {TEMPERATURE}  |  ROWS: {from_row} to {to_row} ({to_row - from_row + 1} sentences)  |  AVG GROUND TRUTH: {metrics_df['Ground_Truth_Count'].mean():.2f} terms/sentence"""

ax1.text(0.5, 0.5, config_text, ha='center', va='center',
         fontsize=11, weight='bold', family='monospace',
         bbox=dict(boxstyle='round', facecolor='#ecf0f1', edgecolor='#34495e', linewidth=2))

# ============================================================
# SECTION 2: Metrics Table
# ============================================================
ax2 = fig.add_subplot(gs[1])
ax2.axis('tight')
ax2.axis('off')

table_data = [
    ['Metric', 'Prompt A', 'Prompt B', 'Prompt C'],
    ['Precision', f"{avg_metrics['Prompt A'][0]:.4f}", f"{avg_metrics['Prompt B'][0]:.4f}", f"{avg_metrics['Prompt C'][0]:.4f}"],
    ['Recall', f"{avg_metrics['Prompt A'][1]:.4f}", f"{avg_metrics['Prompt B'][1]:.4f}", f"{avg_metrics['Prompt C'][1]:.4f}"],
    ['F1-Score', f"{avg_metrics['Prompt A'][2]:.4f}", f"{avg_metrics['Prompt B'][2]:.4f}", f"{avg_metrics['Prompt C'][2]:.4f}"]
]

cell_colors = [
    ['#3498db', '#3498db', '#3498db', '#3498db'],
    ['#f8f9fa', 'white', 'white', 'white'],
    ['#f8f9fa', 'white', 'white', 'white'],
    ['#f8f9fa'] + ['#2ecc71' if i == best_f1_idx else 'white' for i in range(3)]
]

table = ax2.table(cellText=table_data, cellLoc='center', loc='center',
                 cellColours=cell_colors, bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(13)
table.scale(1, 3)

for i in range(4):
    table[(0, i)].set_text_props(weight='bold', color='white')
for i in range(1, 4):
    table[(i, 0)].set_text_props(weight='bold')

# ============================================================
# SECTION 3: FULL PROMPTS (NO TRUNCATION)
# ============================================================
ax3 = fig.add_subplot(gs[2])
ax3.axis('off')

from textwrap import fill

# Get full prompts - NO TRUNCATION
prompt_a_full = fill(prompt_A, width=130)
prompt_b_full = fill(prompt_B, width=130)
prompt_c_full = fill(prompt_C, width=130)

prompts_text = f"""PROMPT A:
{prompt_a_full}

PROMPT B:
{prompt_b_full}

PROMPT C:
{prompt_c_full}"""

ax3.text(0.02, 0.98, prompts_text, ha='left', va='top',
         fontsize=8, family='monospace',
         bbox=dict(boxstyle='round', facecolor='#fef9e7', edgecolor='#f39c12', linewidth=2, pad=1))

# ============================================================
# SECTION 4: Footer
# ============================================================
ax4 = fig.add_subplot(gs[3])
ax4.axis('off')

footer_text = f"🏆 BEST PERFORMING: Prompt {chr(65 + best_f1_idx)} (F1-Score = {avg_metrics[['Prompt A', 'Prompt B', 'Prompt C'][best_f1_idx]][2]:.4f})"
ax4.text(0.5, 0.5, footer_text, ha='center', va='center',
         fontsize=14, weight='bold', color='white',
         bbox=dict(boxstyle='round', facecolor='#27ae60', edgecolor='#229954', linewidth=2))

# Main title
fig.suptitle('Evaluation Metrics Summary', fontsize=18, weight='bold', y=0.98)

# Save
img2_filename = f"evaluation_summary_row_{from_row}_to_{to_row}.png"
plt.savefig(img2_filename, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(f"✓ Image 2 saved: {img2_filename}")

# ============================================================
# Summary
# ============================================================
print("\n" + "="*70)
print("📊 IMAGES CREATED SUCCESSFULLY")
print("="*70)
print(f"Image 1 (Data Table): {img1_filename}")
print(f"Image 2 (Evaluation Summary): {img2_filename}")
print(f"\n🏆 Best Prompt: Prompt {chr(65 + best_f1_idx)} with F1-Score = {avg_metrics[['Prompt A', 'Prompt B', 'Prompt C'][best_f1_idx]][2]:.4f}")